# Ticket T-106: Baseline Model Training & Evaluation

This notebook implements and benchmarks baseline regression models for predicting NYC yellow taxi **fare amount** ($) and **trip duration** (minutes) for ticket **T-106**.

### Architectural Decision: Single-Output vs Multi-Output Models
**Decision**: We train two separate single-output models (one for `fare_amount`, one for `duration_minutes`) rather than a single multi-output model.

**Justification**:
1. **Loss Surface & Error Distribution**: Fares ($) and durations (minutes) operate on distinct financial and temporal scales with different error structures.
2. **Independent Hyperparameter Tuning**: Single-output models allow optimizing tree depth, regularization, and leaf sizes independently for price vs time.
3. **Interpretability**: Feature importances and residual diagnostics remain clear and decoupled per target.

### Acceptance Criteria Verified:
1. **Trivial Baseline**: Benchmarks a dummy mean regressor predicting historical training average.
2. **Decision Tree Baseline**: Fits single-output `DecisionTreeRegressor` for both targets.
3. **Comprehensive Metrics**: Reports MAE, RMSE, MAPE, and R² on unseen temporal test set.
4. **Performance Benchmarking**: Records training time (seconds) and single-row inference latency (milliseconds).

In [1]:
import os
import sys
import pickle
import time
from pathlib import Path

# Ensure project root directory is in sys.path when running from notebooks/ directory
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.config import (
    TRAIN_CLEANED_PATH,
    TEST_CLEANED_PATH,
    ALLOWED_FEATURES,
    MODELS_DIR
)
from src.train import (
    calculate_metrics,
    measure_inference_time,
    train_and_evaluate_baselines
)

## 1. Execute Baseline Training Pipeline
Runs `train_and_evaluate_baselines()` to fit the Trivial Mean Regressor and Decision Tree Regressor across both targets.

In [2]:
baseline_output = train_and_evaluate_baselines(save_models=True)
summary_df = baseline_output["summary_table"]
display(summary_df)

2026-08-12 15:48:39,455 - INFO - --- Starting Baseline Model Training (T-106) ---
2026-08-12 15:48:39,455 - INFO - Loading cleaned train data from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/dataset/train_cleaned.parquet...
2026-08-12 15:48:39,611 - INFO - Loading cleaned test data from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/dataset/test_cleaned.parquet...
2026-08-12 15:48:39,654 - INFO - Loading existing feature pipeline from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/models/feature_pipeline.pkl...
2026-08-12 15:48:41,229 - INFO - --- Training Trivial Mean Baseline ---
2026-08-12 15:48:41,316 - INFO - --- Training Decision Tree Regressors ---
2026-08-12 15:49:18,603 - INFO - 
=== Baseline Models Evaluation Summary (Test Set) ===
                          MAE     RMSE   MAPE      R2  train_time_sec  inference_latency_ms
Trivial_Mean_Fare      8.8515  13.5709  77.69 -0.0002          0.0372                0.0077
Trivial_Mean_Dur

,MAE,RMSE,MAPE,R2,train_time_sec,inference_latency_ms
Trivial_Mean_Fare,8.8515,13.5709,77.69,-0.0002,0.0372,0.0077
Trivial_Mean_Duration,9.2712,13.4220,99.09,-0.0002,0.0025,0.0085
DecisionTree_Fare,1.4581,2.8822,12.23,0.9549,18.4237,0.4290
DecisionTree_Duration,3.9406,6.3713,30.31,0.7746,17.8215,0.4006


## 2. Feature Importance Analysis (Decision Tree)
Examines the top predictive features driving fare and duration predictions in the Decision Tree Regressor.

In [3]:
# Load fitted feature pipeline to inspect feature names
with open(os.path.join(MODELS_DIR, "feature_pipeline.pkl"), "rb") as f:
    pipeline = pickle.load(f)

feature_names = pipeline.feature_names_
dt_fare = baseline_output["models"]["dt_fare"]
dt_dur = baseline_output["models"]["dt_dur"]

# Top 10 Features for Fare
fare_importance = pd.Series(dt_fare.feature_importances_, index=feature_names).sort_values(ascending=False)
print("=== Top 10 Features for Fare Amount ($) ===")
display(fare_importance.head(10))

# Top 10 Features for Duration
dur_importance = pd.Series(dt_dur.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\n=== Top 10 Features for Trip Duration (minutes) ===")
display(dur_importance.head(10))

=== Top 10 Features for Fare Amount ($) ===


trip_distance              0.935742
RatecodeID_target_enc      0.024327
RatecodeID                 0.023812
cos_hour                   0.003397
do_lon                     0.002436
DOLocationID_target_enc    0.001609
PULocationID_target_enc    0.001163
sin_hour                   0.001073
do_lat                     0.000993
pickup_day                 0.000775
dtype: float64


=== Top 10 Features for Trip Duration (minutes) ===


trip_distance              0.834931
cos_hour                   0.058957
sin_hour                   0.017253
pickup_hour                0.013537
pickup_day                 0.009662
pickup_dayofweek           0.008154
cos_dayofweek              0.007958
do_lon                     0.007685
do_lat                     0.007477
DOLocationID_target_enc    0.007086
dtype: float64

## 3. Metric Comparison Summary
Comparing baseline performance improvements on the unseen temporal test set.

In [4]:
print("=== Benchmark Key Takeaways ===")
print(f"1. Fare MAE improved from ${summary_df.loc['Trivial_Mean_Fare', 'MAE']:.2f} (Trivial Mean) to ${summary_df.loc['DecisionTree_Fare', 'MAE']:.2f} (Decision Tree).")
print(f"2. Fare R² improved from {summary_df.loc['Trivial_Mean_Fare', 'R2']} to {summary_df.loc['DecisionTree_Fare', 'R2']:.4f}.")
print(f"3. Duration MAE improved from {summary_df.loc['Trivial_Mean_Duration', 'MAE']:.2f} mins to {summary_df.loc['DecisionTree_Duration', 'MAE']:.2f} mins.")
print(f"4. Duration R² improved from {summary_df.loc['Trivial_Mean_Duration', 'R2']} to {summary_df.loc['DecisionTree_Duration', 'R2']:.4f}.")
print(f"5. Single-row inference latency for Decision Tree is ultra-fast: ~{summary_df.loc['DecisionTree_Fare', 'inference_latency_ms']:.3f} ms per request.")

=== Benchmark Key Takeaways ===
1. Fare MAE improved from $8.85 (Trivial Mean) to $1.46 (Decision Tree).
2. Fare R² improved from -0.0002 to 0.9549.
3. Duration MAE improved from 9.27 mins to 3.94 mins.
4. Duration R² improved from -0.0002 to 0.7746.
5. Single-row inference latency for Decision Tree is ultra-fast: ~0.429 ms per request.
